In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D
from tensorflow.keras.optimizers import Adam


In [60]:
import cv2

video_path = "dog_video.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if not ret:
    print("Failed to read frame")
else:
    print(frame.shape)

(432, 768, 3)


In [61]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

model = MobileNet(weights="imagenet")

img = cv2.resize(frame, (224,224))
img = image.img_to_array(img)
img = np.expand_dims(img, axis=0)
img = preprocess_input(img)

results = model.predict(img)

print(decode_predictions(results, top=5)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
[('n03207941', 'dishwasher', np.float32(0.16281188)), ('n02815834', 'beaker', np.float32(0.14054695)), ('n04125021', 'safe', np.float32(0.113481104)), ('n03733805', 'measuring_cup', np.float32(0.08279186)), ('n02906734', 'broom', np.float32(0.056719374))]


In [62]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model(frame)

results[0].show()


0: 384x640 1 chair, 79.8ms
Speed: 4.4ms preprocess, 79.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)


In [63]:
while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame)

cap.release()


0: 384x640 1 chair, 78.4ms
Speed: 5.2ms preprocess, 78.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 72.3ms
Speed: 3.4ms preprocess, 72.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 72.4ms
Speed: 3.6ms preprocess, 72.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 70.2ms
Speed: 3.5ms preprocess, 70.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 69.4ms
Speed: 3.5ms preprocess, 69.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 69.8ms
Speed: 3.6ms preprocess, 69.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 69.1ms
Speed: 3.8ms preprocess, 69.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 chair, 72.0ms
Speed: 4.2ms preprocess, 72.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)


In [64]:
imu = pd.read_csv("collar_imu.csv")

In [65]:
import numpy as np
import pandas as pd

imu = pd.read_csv("collar_imu.csv")

imu["mag"] = np.sqrt(
    imu["accel_x"]**2 +
    imu["accel_y"]**2 +
    imu["accel_z"]**2
)

In [66]:
print(imu.head())

   timestamp_ms  accel_x  accel_y  accel_z   gyro_x   gyro_y   gyro_z  \
0             0   0.9489   0.7106   0.6815  46.7733  28.8680  17.7830   
1            10   1.9858   0.7881  -0.0269  30.5563  44.9629   7.3114   
2            20   1.3979   0.5114   0.8332  42.3856  41.8207  17.3891   
3            30   1.2788   0.6822   1.0544  14.9691  58.7830  21.4190   
4            40   1.7132   0.9421   0.6509  37.8887  33.9745   8.6557   

        mag  
0  1.367408  
1  2.136639  
2  1.705836  
3  1.792341  
4  2.060650  


In [67]:
print(imu.columns)

Index(['timestamp_ms', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y',
       'gyro_z', 'mag'],
      dtype='object')


In [68]:
print(imu.columns)

Index(['timestamp_ms', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y',
       'gyro_z', 'mag'],
      dtype='object')


In [69]:
imu["window"] = imu["timestamp_ms"] // 500

imu_summary = (
    imu.groupby("window")
       .agg(
           timestamp_ms=("timestamp_ms", "first"),
           mean_mag=("mag", "mean")
       )
       .reset_index(drop=True)
)

print(imu_summary.head())

   timestamp_ms  mean_mag
0             0  1.722176
1           500  1.932631
2          1000  1.858098
3          1501  1.738056
4          2001  1.792738


In [70]:
threshold = 1.8

imu_summary["imu_activity"] = np.where(
    imu_summary["mean_mag"] > threshold,
    "Active",
    "Static"
)

print(imu_summary)

    timestamp_ms  mean_mag imu_activity
0              0  1.722176       Static
1            500  1.932631       Active
2           1000  1.858098       Active
3           1501  1.738056       Static
4           2001  1.792738       Static
5           2502  1.867544       Active
6           3002  1.850451       Active
7           3502  1.930826       Active
8           4003  1.954604       Active
9           4503  1.847539       Active
10          5004  1.898060       Active
11          5504  1.890671       Active
12          6005  0.907606       Static
13          6505  0.909202       Static
14          7005  0.909657       Static
15          7506  0.908670       Static
16          8006  0.909956       Static
17          8507  0.905276       Static
18          9007  0.910830       Static
19          9507  0.907554       Static
20         10008  0.912260       Static
21         10508  0.910001       Static
22         11009  0.911052       Static
23         11509  0.910355       Static


In [71]:
vision_activity = "Static"
vision_conf = 0.55

In [72]:
imu_mag = imu_summary.loc[0, "mean_mag"]

threshold = 1.8

if imu_mag > threshold:
    final = "Active"
else:
    final = vision_activity

print(final)

Static


In [73]:
max_mag = imu_summary["mean_mag"].max()

imu_conf = imu_mag / max_mag

confidence = max(vision_conf, imu_conf)

print(confidence)

0.8810867155281915


In [74]:
#json

In [75]:
import json

output = [
    {
        "timestamp_ms": 0,
        "activity": "Static",
        "confidence": 0.82
    },
    {
        "timestamp_ms": 500,
        "activity": "Active",
        "confidence": 0.91
    }
]

with open("timeline.json", "w") as f:
    json.dump(output, f, indent=4)

print("timeline.json created successfully!")

timeline.json created successfully!


In [76]:
[
  {
    "timestamp_ms": 0,
    "activity": "Static",
    "confidence": 0.82
  },
  {
    "timestamp_ms": 500,
    "activity": "Active",
    "confidence": 0.91
  },
  {
    "timestamp_ms": 1000,
    "activity": "Active",
    "confidence": 0.95
  }
]

[{'timestamp_ms': 0, 'activity': 'Static', 'confidence': 0.82},
 {'timestamp_ms': 500, 'activity': 'Active', 'confidence': 0.91},
 {'timestamp_ms': 1000, 'activity': 'Active', 'confidence': 0.95}]

In [77]:
output = []

In [78]:
timestamp = 0
activity = "Static"
confidence = 0.82

In [79]:
output.append({
    "timestamp_ms": timestamp,
    "activity": activity,
    "confidence": round(confidence, 2)
})

In [80]:
number_of_samples = 12 * 2
print(number_of_samples)   # 24

24


In [81]:
output = []

number_of_samples = 24

for i in range(number_of_samples):
    timestamp = i * 500          # 0, 500, 1000, ...
    activity = "Static"          # Replace with your predicted activity
    confidence = 0.80            # Replace with your computed confidence

    output.append({
        "timestamp_ms": timestamp,
        "activity": activity,
        "confidence": round(confidence, 2)
    })

import json

with open("timeline.json", "w") as f:
    json.dump(output, f, indent=4)

print("timeline.json created successfully!")

timeline.json created successfully!


In [82]:
[
    {
        "timestamp_ms": 0,
        "activity": "Static",
        "confidence": 0.8
    },
    {
        "timestamp_ms": 500,
        "activity": "Static",
        "confidence": 0.8
    },
    {
        "timestamp_ms": 1000,
        "activity": "Static",
        "confidence": 0.8
    }
]

[{'timestamp_ms': 0, 'activity': 'Static', 'confidence': 0.8},
 {'timestamp_ms': 500, 'activity': 'Static', 'confidence': 0.8},
 {'timestamp_ms': 1000, 'activity': 'Static', 'confidence': 0.8}]